In [ ]:
"""
Concepts:

1. Leaf-wise Growth
2. GOSS
3. EFB
4. num_leaves
5. Early Stopping
6. Feature Importance
7. Calibration
8. SHAP Explainability
"""

# =====================================================
# IMPORTS
# =====================================================

import warnings
warnings.filterwarnings("ignore")
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import (train_test_split,cross_val_score,RandomizedSearchCV,learning_curve)
from sklearn.metrics import (accuracy_score,precision_score,recall_score,f1_score,roc_auc_score,confusion_matrix,classification_report,roc_curve,precision_recall_curve)
from sklearn.calibration import calibration_curve
from sklearn.inspection import permutation_importance
from lightgbm import LGBMClassifier

In [ ]:
# =====================================================
# STEP 1 : DATA UNDERSTANDING
# =====================================================

data = load_breast_cancer()
df = pd.DataFrame(data.data,columns=data.feature_names)
df["target"] = data.target
print(df.head())
print(df.info())
print(df.describe())
print(df["target"].value_counts())

In [ ]:
# =====================================================
# STEP 2 : EDA
# =====================================================

print(df.isnull().sum())
print(df.duplicated().sum())

In [ ]:
# =====================================================
# STEP 3 : TRAIN / VALIDATION / TEST
# =====================================================

X = df.drop("target", axis=1)
y = df["target"]
X_temp, X_test, y_temp, y_test = train_test_split(X,y,test_size=0.15,random_state=42,stratify=y)
X_train, X_val, y_train, y_val = train_test_split(X_temp,y_temp,test_size=0.1765,random_state=42,stratify=y_temp)

# =====================================================
# STEP 4 : PREPROCESSING
# =====================================================

# LightGBM does NOT require scaling

In [ ]:
# =====================================================
# STEP 5 : BASELINE MODEL
# =====================================================

lgbm = LGBMClassifier(n_estimators=300,learning_rate=0.05,num_leaves=31,random_state=42)
lgbm.fit(X_train,y_train)

In [ ]:
# =====================================================
# STEP 6 : VALIDATION METRICS
# =====================================================

val_pred = lgbm.predict(X_val)
val_prob = lgbm.predict_proba(X_val)[:,1]
print("Accuracy:",accuracy_score(y_val,val_pred))
print("Precision:",precision_score(y_val,val_pred))
print("Recall:",recall_score(y_val,val_pred))
print("F1:",f1_score(y_val,val_pred))
print("ROC-AUC:",roc_auc_score(y_val,val_prob))

In [ ]:
# =====================================================
# STEP 7 : CROSS VALIDATION
# =====================================================

cv_scores = cross_val_score(lgbm,X_train,y_train,cv=5,scoring="roc_auc",n_jobs=-1)
print(cv_scores.mean())
print(cv_scores.std())

In [ ]:
# =====================================================
# STEP 8 : HYPERPARAMETER TUNING
# =====================================================

param_dist = {
    "n_estimators":[200,300,500],
    "learning_rate":[0.01,0.05,0.1],
    "num_leaves":[15,31,63,127],
    "max_depth":[-1,3,5,7],
    "min_child_samples":[10,20,50],
    "subsample":[0.7,0.8,1.0],
    "colsample_bytree":[0.7,0.8,1.0],
    "reg_alpha":[0,0.1,1],
    "reg_lambda":[1,5,10]}

random_search = RandomizedSearchCV(LGBMClassifier(random_state=42),param_dist,n_iter=30,cv=5,scoring="roc_auc",random_state=42,n_jobs=-1)
random_search.fit(X_train,y_train)
best_model = random_search.best_estimator_
print(random_search.best_params_)

In [ ]:
# =====================================================
# STEP 9 : VALIDATION RECHECK
# =====================================================

val_prob = best_model.predict_proba(X_val)[:,1]
val_auc = roc_auc_score(y_val,val_prob)
print(val_auc)

In [ ]:
# =====================================================
# STEP 10 : TRAIN VS VALIDATION
# =====================================================

train_prob = best_model.predict_proba(X_train)[:,1]
train_auc = roc_auc_score(y_train,train_prob)
print(train_auc)
print(val_auc)

In [ ]:
# =====================================================
# STEP 11 : MODEL ANALYSIS
# =====================================================

# FEATURE IMPORTANCE

importance_df = pd.DataFrame({"Feature":X.columns,"Importance":best_model.feature_importances_})
importance_df = importance_df.sort_values(by="Importance",ascending=False)
print(importance_df.head(15))

# PERMUTATION IMPORTANCE

perm = permutation_importance(best_model,X_val,y_val,random_state=42)
print(perm.importances_mean)

# CONFUSION MATRIX

cm = confusion_matrix(y_val,best_model.predict(X_val))
sns.heatmap(cm,annot=True,fmt='d')
plt.show()

# ROC CURVE
fpr,tpr,_ = roc_curve(y_val,val_prob)
plt.plot(fpr,tpr)
plt.plot([0,1],[0,1])
plt.show()

# PR CURVE

precision, recall, _ = precision_recall_curve(y_val,val_prob)
plt.plot(recall,precision)
plt.show()

# CALIBRATION CURVE

prob_true, prob_pred = calibration_curve(y_val,val_prob,n_bins=10)
plt.plot(prob_pred,prob_true,marker='o')
plt.plot([0,1],[0,1])
plt.show()

# NUM LEAVES ANALYSIS

leaves = [15,31,63,127]
scores = []

for leaf in leaves:
    model = LGBMClassifier(num_leaves=leaf,random_state=42)
    model.fit(X_train,y_train)
    prob = model.predict_proba(X_val)[:,1]
    scores.append(roc_auc_score(y_val,prob))

plt.plot(leaves,scores,marker='o')
plt.title("Num Leaves Analysis")
plt.show()

# LEARNING CURVE

train_sizes, train_scores, val_scores = learning_curve(best_model,X_train,y_train,cv=5,scoring="roc_auc",n_jobs=-1)
plt.plot(train_sizes,np.mean(train_scores,axis=1),label="Train")
plt.plot(train_sizes,np.mean(val_scores,axis=1),label="Validation")
plt.legend()
plt.show()


In [ ]:
# =====================================================
# STEP 12 : FINAL MODEL
# =====================================================

final_model = best_model

In [ ]:
# =====================================================
# STEP 13 : TEST EVALUATION
# =====================================================

test_pred = final_model.predict(X_test)
test_prob = final_model.predict_proba(X_test)[:,1]
print(classification_report(y_test,test_pred))
print(roc_auc_score(y_test,test_prob))

In [ ]:
# =====================================================
# STEP 14 : DEPLOYMENT READINESS
# =====================================================

print("CV:",cv_scores.mean())
print("Validation:",val_auc)
print("Test:",roc_auc_score(y_test,test_prob))


# =====================================================
# STEP 15 : INTERVIEW QUESTIONS
# =====================================================

"""
1. What is LightGBM?
2. Difference between XGBoost and LightGBM?
3. What is leaf-wise growth?
4. Why is LightGBM faster?
5. What is GOSS?
6. What is EFB?
7. What is num_leaves?
8. Why num_leaves is important?
9. What is min_child_samples?
10. Why LightGBM overfits easily?
11. Difference between level-wise and leaf-wise?
12. What is feature_fraction?
13. What is bagging_fraction?
14. What is regularization?
15. What is early stopping?
16. Why LightGBM is popular?
17. When should XGBoost be preferred?
18. When should LightGBM be preferred?
19. LightGBM limitations?
20. Explain LightGBM end-to-end.
"""